# NSE Dividend PDF Downloader (Colab) — v2
Downloads **~52,030** dividend announcement documents (PDFs + ZIPs) from NSE → Google Drive.

- Source CSV  : `nse_dividend_pdfs.csv` — 58,345 rows, 52,030 downloadable URLs
- ZIPs are auto-extracted; only PDFs inside are kept
- Full resume support — re-run any time, nothing is re-downloaded
- Progress saved every 100 files **and** every 4 minutes

**v2 fixes**: Session cookies, rate limiting, Colab-IP fallback,
bad-content detection, Drive-scan caching, diagnostics.

**Run Cell 0 → 1 → 2 → 2.5 → 3 → 4**

In [41]:
# CELL 0: Keepalive — prevents Colab from disconnecting mid-run
import threading, time
def _ka():
    while True: time.sleep(55); _ = 1 + 1
if not any(t.name == 'keepalive' for t in threading.enumerate()):
    threading.Thread(target=_ka, name='keepalive', daemon=True).start()
    print('✅ Keepalive started')
else:
    print('✅ Keepalive already running')

✅ Keepalive already running


In [42]:
# CELL 1: Mount Drive (skips if already mounted)
import os, shutil, time
from google.colab import drive
MOUNT = '/content/drive'
already_ok = False
try:
    tp = os.path.join(MOUNT, 'MyDrive')
    if os.path.isdir(tp) and os.listdir(tp):
        already_ok = True
except:
    pass
if already_ok:
    print('✅ Drive already mounted — skipping.')
else:
    if os.path.isdir(MOUNT):
        try: shutil.rmtree(MOUNT, ignore_errors=True)
        except: pass
    for attempt in range(1, 6):
        try:
            drive.mount(MOUNT)
            print('✅ Drive mounted!')
            break
        except Exception as e:
            if 'already contain' in str(e): shutil.rmtree(MOUNT, ignore_errors=True)
            if attempt < 5:
                print(f'Mount {attempt}/5 failed: {e}, retrying in {10*attempt}s')
                time.sleep(10 * attempt)
            else:
                raise

✅ Drive already mounted — skipping.


In [43]:
# CELL 2: Set paths — hardcoded for instant startup (no slow os.walk)
import os

# ── Paths (match your Drive layout exactly) ───────────────────────────────────
MANIFEST_CSV = '/content/drive/MyDrive/ArthLLM-2B/NSE DIVIDENT/nse_dividend_pdfs.csv'
# Output PDFs land here, one sub-folder per symbol:
#   NSE DIVIDENT/NSE_Dividend_PDFs/<SYMBOL>/

if not os.path.exists(MANIFEST_CSV):
    raise FileNotFoundError(
        f'CSV not found:\n  {MANIFEST_CSV}\n'
        'Check your Drive path and folder name (note the space in "NSE DIVIDENT").'
    )

BASE    = os.path.dirname(MANIFEST_CSV)
OUT_DIR = os.path.join(BASE, 'NSE_Dividend_PDFs')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'✅ Manifest : {MANIFEST_CSV}')
print(f'📥 Output   : {OUT_DIR}')

✅ Manifest : /content/drive/MyDrive/ArthLLM-2B/NSE DIVIDENT/nse_dividend_pdfs.csv
📥 Output   : /content/drive/MyDrive/ArthLLM-2B/NSE DIVIDENT/NSE_Dividend_PDFs


In [44]:
# CELL 2.5: NSE Connectivity Diagnostic
# Run AFTER mounting Drive + setting paths so we can read the CSV.
# Tests whether this runtime can reach NSE before wasting hours.
import requests, time, csv

DIAG_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Referer': 'https://www.nseindia.com/',
    'Connection': 'keep-alive',
}

# ── Find a real PDF URL from the manifest ─────────────────────────────────────
test_pdf_url = None
with open(MANIFEST_CSV, 'r', encoding='utf-8', errors='replace') as f:
    reader = csv.DictReader(f)
    for row in reader:
        u = (row.get('pdf_url') or '').strip()
        if u and u.startswith('http') and u.lower().endswith('.pdf'):
            test_pdf_url = u
            break
if test_pdf_url:
    print(f'Test URL: {test_pdf_url[:90]}...')
else:
    print('⚠️  No .pdf URL found in CSV')

print('─── NSE Connectivity Diagnostic ───')

# Test 1: Can we reach nsearchives at all?
try:
    r = requests.head('https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv',
                      headers=DIAG_HEADERS, timeout=15, allow_redirects=True)
    print(f'  1. nsearchives reachability → HTTP {r.status_code} (expected 200)')
except Exception as e:
    print(f'  1. nsearchives reachability → FAILED: {e}')

# Test 2: Try a real PDF URL from the CSV
if test_pdf_url:
    try:
        sess = requests.Session()
        try:
            sess.get('https://www.nseindia.com/', headers=DIAG_HEADERS, timeout=10)
            time.sleep(1)
        except:
            pass  # May fail on Colab, that's fine
        r2 = sess.get(test_pdf_url, headers=DIAG_HEADERS, timeout=15)
        first_4 = r2.content[:4] if r2.status_code == 200 else b''
        print(f'  2. Test PDF → HTTP {r2.status_code} | first 4 bytes: {first_4}')
        if first_4 == b'%PDF':
            print('  ✅ Session works — PDF bytes confirmed. Proceed!')
        elif r2.status_code == 200 and b'<html' in r2.content[:1024].lower():
            print('  ⚠️  Got HTML instead of PDF — NSE may be blocking this IP.')
            print('     Downloads may still work for some files. Try running locally if failure rate is high.')
        elif r2.status_code == 404:
            print('  ⚠️  404 — this specific file may have been removed from NSE.')
            print('     This does NOT mean all downloads will fail. Proceed and check results.')
        elif r2.status_code == 403:
            print('  ❌ 403 Forbidden — NSE is actively blocking this IP.')
            print('     Consider running locally or using a residential proxy.')
        else:
            print(f'  ⚠️  Unexpected response. Status={r2.status_code}, size={len(r2.content)}')
        del sess
    except Exception as e:
        print(f'  2. Test PDF → FAILED: {e}')
else:
    print('  2. Skipped (no test URL available)')

print('───────────────────────────────────')
del DIAG_HEADERS  # cleanup

Test URL: https://nsearchives.nseindia.com/corporate/ITNLBookClosureNotice2013.pdf...
─── NSE Connectivity Diagnostic ───
  1. nsearchives reachability → HTTP 200 (expected 200)
  2. Test PDF → HTTP 200 | first 4 bytes: b'%PDF'
  ✅ Session works — PDF bytes confirmed. Proceed!
───────────────────────────────────


In [45]:
# CELL 3: Fast Resume — two-way sync manifest ↔ disk (with caching)
import os, re, time, json
import pandas as pd
from tqdm.notebook import tqdm

# ── Load CSV ──────────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'CORRUPT CSV: {e}')
if len(df) == 0: raise RuntimeError('CSV has 0 rows')
print(f'Loaded {len(df):,} rows')

# ── Normalise dl_status column ────────────────────────────────────────────────
DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

# ── Mark rows with no URL so they're never attempted ─────────────────────────
no_url_mask = df['pdf_url'].isna() | (df['pdf_url'].astype(str).str.strip().isin(['-', '', 'nan']))
df.loc[no_url_mask, 'dl_status'] = 'no_url'
print(f'  Rows with no URL : {no_url_mask.sum():,}')

# ── Scan existing PDFs on disk (BUG 5 FIX: cached + timed) ────────────────────
CACHE_FILE = '/tmp/existing_files_cache.txt'
CACHE_MAX_AGE = 30 * 60  # 30 minutes

use_cache = False
if os.path.exists(CACHE_FILE):
    cache_age = time.time() - os.path.getmtime(CACHE_FILE)
    if cache_age < CACHE_MAX_AGE:
        use_cache = True
        print(f'Loading cached file list ({cache_age/60:.1f} min old)...')

if use_cache:
    with open(CACHE_FILE, 'r') as f:
        existing = set(line.strip() for line in f if line.strip())
    print(f'Found {len(existing):,} PDFs from cache')
else:
    print('Scanning disk for existing files ( PDFs, TIFs, JPGs )...')
    scan_start = time.time()
    existing = set()
    if os.path.exists(OUT_DIR):
        for root, dirs, files in os.walk(OUT_DIR):
            for x in files:
                if x.lower().endswith(('.pdf', '.tif', '.tiff', '.jpg', '.jpeg', '.png', '.doc', '.docx')):
                    existing.add(x)
    scan_elapsed = time.time() - scan_start
    print(f'Found {len(existing):,} valid docs already on disk (scan took {scan_elapsed:.1f}s)')
    if scan_elapsed > 180:
        print('⚠️  Drive scan slow — consider running Cell 3 only once '
              'and not clearing runtime between runs')
    # Save cache
    with open(CACHE_FILE, 'w') as f:
        f.write('\n'.join(sorted(existing)))
    print(f'  Cached to {CACHE_FILE}')

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def expected_fname(row):
    sym  = safe(row.get('symbol', 'UNKNOWN'))
    seq  = safe(str(row.get('seq_id', '')).split('.')[0])
    url  = str(row.get('pdf_url', ''))
    base = os.path.splitext(os.path.basename(url))[0][:50]
    return f'{sym}_{seq}_{base}.pdf'

# ── Two-way sync: manifest ↔ disk ─────────────────────────────────────────────
updated = 0
for i in tqdm(df.index, desc='Syncing manifest'):
    if df.at[i, 'dl_status'] == 'no_url': continue
    cur = df.at[i, 'dl_status']
    fn  = expected_fname(df.loc[i])
    # Let's consider it done if any extension exists
    base = os.path.splitext(fn)[0]
    fe = any(f"{base}{ext}" in existing for ext in ['.pdf', '.tif', '.tiff', '.jpg', '.jpeg', '.png', '.doc', '.docx'])
    if cur == 'done' and not fe:  df.at[i, 'dl_status'] = 'pending'; updated += 1
    elif cur != 'done' and fe:    df.at[i, 'dl_status'] = 'done';    updated += 1

if updated > 0:
    df.to_csv(MANIFEST_CSV, index=False, quoting=1)
    print(f'Updated {updated:,} rows in manifest')
else:
    print('Manifest already in sync — no changes needed')

print('\nStatus summary:')
for s, c in df['dl_status'].value_counts().items():
    print(f'  {s:15s}: {c:,}')

Loaded 13,449 rows
  Rows with no URL : 6,264
Loading cached file list (26.9 min old)...
Found 32,186 PDFs from cache


Syncing manifest:   0%|          | 0/13449 [00:00<?, ?it/s]

Updated 18 rows in manifest

Status summary:
  done           : 7,051
  no_url         : 6,264
  unzip_fail     : 76
  pending        : 33
  http_404       : 15
  fail           : 10


In [46]:
# CELL 4: Download NSE Dividend Docs — all bugs fixed
import pandas as pd, requests, os, re, time, threading, zipfile, io, random
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ── Tuning (BUG 3 FIX: reduced workers) ───────────────────────────────────────
WORKERS     = 3     # NSE rate limit ~3 req/s; DO NOT raise
SAVE_EVERY  = 100   # flush manifest every N completions
TIMER_SAVE  = 240   # also flush every 4 minutes
TIMEOUT     = 45    # seconds per request
MAX_RETRIES = 5     # retries per URL
RETRY_CODES = {429, 500, 502, 503, 504}  # codes worth retrying
REWARM_EVERY = 100  # re-warm session every N downloads

# ── Headers (BUG 1 FIX: added NSE-required headers) ───────────────────────────
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept'          : 'application/pdf,application/zip,*/*',
    'Accept-Encoding' : 'gzip, deflate, br',
    'Accept-Language'  : 'en-US,en;q=0.9',
    'Connection'      : 'keep-alive',
    'Referer'         : 'https://www.nseindia.com/',
    'Sec-Fetch-Dest'  : 'document',
    'Sec-Fetch-Mode'  : 'navigate',
    'Sec-Fetch-Site'  : 'same-origin',
}

# ── BUG 1 FIX: Shared Session + Cookie Warm-Up ────────────────────────────────
sess = requests.Session()
sess.headers.update(HEADERS)

def warm_up_session():
    """Warm up session cookies. Handles Colab IP blocks gracefully (BUG 2)."""
    print(f'[{datetime.now():%H:%M:%S}] Warming up NSE session...')
    # Try www.nseindia.com first
    try:
        r = sess.get('https://www.nseindia.com/', timeout=10)
        print(f'  www.nseindia.com → {r.status_code}')
        r.raise_for_status() # Force exception on 403
        time.sleep(2)
        r2 = sess.get(
            'https://www.nseindia.com/companies-listing/corporate-filings-announcements',
            timeout=10
        )
        print(f'  corporate-filings → {r2.status_code}')
        r2.raise_for_status()
        time.sleep(2)
        print('  ✅ Session warmed via www.nseindia.com')
        return True
    except Exception as e:
        print(f'  ⚠️  www.nseindia.com failed ({e}) — triggering fallback...')

    # BUG 2 FIX: Fallback to nsearchives directly
    try:
        # Use a real file instead of the root / which naturally returns 404
        r = sess.get('https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv', timeout=10)
        print(f'  nsearchives fallback → {r.status_code}')
        r.raise_for_status()
        time.sleep(2)
        print('  ✅ Session warmed via nsearchives fallback')
        return True
    except Exception as e:
        print(f'  ⚠️  nsearchives fallback failed ({e})')

    print('  ❌ WARNING: NSE may be blocking this server IP. '
          'Downloads may fail. Consider running locally or using a residential proxy.')
    return False

warm_up_session()

# ── BUG 3 FIX: Rate limiting semaphore ────────────────────────────────────────
_semaphore = threading.Semaphore(3)

# ── BUG 4 FIX: bad_content tracking ───────────────────────────────────────────
_recent_results = []  # ring buffer of last 20 results
_recent_lock = threading.Lock()
FAILED_LOG = os.path.join(os.path.dirname(MANIFEST_CSV), 'failed_responses.log')
FAILED_CSV = os.path.join(os.path.dirname(MANIFEST_CSV), 'failed_urls.csv')

def _track_result(result_type):
    """Track recent results for bad_content detection."""
    with _recent_lock:
        _recent_results.append(result_type)
        if len(_recent_results) > 20:
            _recent_results.pop(0)

def _should_pause_and_rewarm():
    """If >50% of last 20 requests are bad_content, we need to pause."""
    with _recent_lock:
        if len(_recent_results) < 10:
            return False
        bad = sum(1 for r in _recent_results if r == 'bad_content')
        return bad > len(_recent_results) * 0.5

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe(s, mx=80):
    return re.sub(r'[<>:"/\\|?*\s]+', '_', str(s).strip())[:mx]

def is_valid_pdf(data: bytes) -> bool:
    """True if bytes look like a real PDF (not an HTML error page)."""
    if not data or len(data) < 64:
        return False
    if data[:4] == b'%PDF':
        return True
    head = data[:1024].lower()
    if b'<html' in head or b'<!doctype' in head or b'<title>' in head:
        return False
    return len(data) > 4096  # large binary with no HTML markers → likely valid

def extract_docs_from_zip(zip_bytes: bytes, dest_dir: str, base_name: str) -> list:
    """Extract ANY valid document file from a ZIP (PDFs, TIFFs, JPGs, DOCX)."""
    VALID_EXTS = ('.pdf', '.tif', '.tiff', '.jpg', '.jpeg', '.png', '.doc', '.docx')
    saved = []
    try:
        # Check if the zip file is completely invalid/corrupted (e.g. truncated)
        # Sometimes Python's zipfile module fails on ZIPs that are incomplete.
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            members = zf.namelist()
            valid_members = [m for m in members if m.lower().endswith(VALID_EXTS)]

            # Handle One level of nested ZIPs
            if not valid_members:
                for m in members:
                    if m.lower().endswith('.zip'):
                        try:
                            inner = zf.read(m)
                            saved += extract_docs_from_zip(inner, dest_dir, base_name + '_inner')
                        except Exception:
                            continue
                return saved

            os.makedirs(dest_dir, exist_ok=True)
            for member in valid_members:
                try:
                    doc_data = zf.read(member)
                    _, original_ext = os.path.splitext(member)
                    # Some NSE zips contain the same file twice, use base name + original extension
                    fname = f'{base_name}{original_ext}'
                    # check if already exists to prevent overwrite from same zip if duplicate
                    if fname in saved:
                        fname = f'{base_name}_{len(saved)}{original_ext}'
                        
                    out_path = os.path.join(dest_dir, fname)
                    with open(out_path, 'wb') as f:
                        f.write(doc_data)
                    saved.append(fname)
                except Exception as e:
                    pass
    except Exception as e:
        # If the python zipfile module legitimately crashes reading the bytes:
        try:
            with open(FAILED_LOG, 'a', encoding='utf-8', errors='replace') as logf:
                logf.write(f'\n--- {datetime.now():%Y-%m-%d %H:%M:%S} | CORRUPT ZIP ARCHIVE ---\n')
                logf.write(f'Exception: {e}\n')
                logf.write(f'First 100 bytes (hex): {zip_bytes[:100].hex()}\n')
        except:
            pass
    return saved

def _get_with_retry(url: str):
    """GET with rate limiting + exponential back-off.
    Returns (Response, None) on success, or (None, str_error) on failure."""
    last_err = 'unknown_error'
    for attempt in range(MAX_RETRIES):
        time.sleep(random.uniform(0.5, 1.5))
        with _semaphore:
            try:
                r = sess.get(url, timeout=TIMEOUT)
                if r.status_code == 200:
                    return r, None
                if r.status_code == 429:
                    wait = 30 + random.uniform(0, 10)
                    time.sleep(wait)
                    last_err = 'http_429'
                    continue
                if r.status_code in RETRY_CODES:
                    wait = min(120, (2 ** attempt) * 5 + random.uniform(0, 3))
                    time.sleep(wait)
                    last_err = f'http_{r.status_code}'
                    continue
                
                # 404 / 403 / other client errors — no point retrying
                return r, None
                
            except requests.exceptions.Timeout as e:
                last_err = 'timeout'
                time.sleep((2 ** attempt) * 5 + random.uniform(0, 3))
            except requests.exceptions.ConnectionError as e:
                last_err = f'conn_err_{type(e).__name__}'
                time.sleep((2 ** attempt) * 5 + random.uniform(0, 3))
            except Exception as e:
                last_err = f'exc_{type(e).__name__}'
                time.sleep((2 ** attempt) * 3 + random.uniform(0, 3))
    return None, last_err

def download_row(url: str, dest_dir: str, base_name: str) -> str:
    """
    Download one row.
    Returns: 'skip' | 'ok' | 'fail' | 'bad_content' | 'http_NNN' | 'conn_err_...' 
    """
    is_zip   = url.lower().endswith('.zip')
    dest_pdf = os.path.join(dest_dir, f'{base_name}.pdf')

    # ── Already downloaded? ───────────────────────────────────────────────────
    if not is_zip:
        if os.path.exists(dest_pdf) and os.path.getsize(dest_pdf) > 1024:
            return 'skip'
    else:
        # For zips, if the base name file exists with ANY valid extension, we're done
        valid_exts = ['.pdf', '.tif', '.tiff', '.jpg', '.jpeg', '.png', '.doc', '.docx']
        if any(os.path.exists(os.path.join(dest_dir, f'{base_name}{ext}')) for ext in valid_exts):
            return 'skip'
        if any(os.path.exists(os.path.join(dest_dir, f'{base_name}_1{ext}')) for ext in valid_exts):
            return 'skip'

    r, err_msg = _get_with_retry(url)
    if r is None:
        _track_result('fail')
        return err_msg  # Return the specific exception message Instead of generic 'fail'
        
    if r.status_code != 200:
        _track_result(f'http_{r.status_code}')
        return f'http_{r.status_code}'

    data = r.content

    if is_zip:
        if not extract_docs_from_zip(data, dest_dir, base_name):
            # ZIP EXTRACTION FAILED OR YIELDED 0 VALID FILES
            head = data[:1024].lower()
            if b'<html' in head or b'<!doctype' in head:
                try:
                    with open(FAILED_LOG, 'a', encoding='utf-8', errors='replace') as f:
                        f.write(f'\n--- {datetime.now():%Y-%m-%d %H:%M:%S} | {url} [ZIP HTML BLOCK] ---\n')
                        f.write(data[:200].decode('utf-8', errors='replace') + '\n')
                except:
                    pass
                _track_result('bad_content')
                return 'bad_content'
            else:
                # Empty zip or contains unsupported files (e.g. .txt)?
                _track_result('unzip_fail')
                return 'unzip_fail'
        else:
            _track_result('ok')
            return 'ok'
    else:
        if not is_valid_pdf(data):
            # BUG 4 FIX: log bad_content response
            try:
                with open(FAILED_LOG, 'a', encoding='utf-8', errors='replace') as f:
                    f.write(f'\n--- {datetime.now():%Y-%m-%d %H:%M:%S} | {url} ---\n')
                    f.write(data[:200].decode('utf-8', errors='replace') + '\n')
            except:
                pass
            _track_result('bad_content')
            return 'bad_content'
        os.makedirs(dest_dir, exist_ok=True)
        with open(dest_pdf, 'wb') as f:
            f.write(data)
        _track_result('ok')
        return 'ok'

# ── Load manifest ─────────────────────────────────────────────────────────────
fsize = os.path.getsize(MANIFEST_CSV) if os.path.exists(MANIFEST_CSV) else 0
if fsize == 0: raise RuntimeError('CORRUPT CSV: 0 bytes')
try:
    df = pd.read_csv(MANIFEST_CSV, dtype={'seq_id': str})
except Exception as e:
    raise RuntimeError(f'Cannot read CSV: {e}')

DONE_VALS = ('done', 'skip', 'true')
if 'dl_status' not in df.columns:
    df['dl_status'] = 'pending'
else:
    df['dl_status'] = df['dl_status'].apply(
        lambda v: 'done' if str(v).strip().lower() in DONE_VALS
                  else (str(v).strip().lower() if pd.notna(v) and str(v).strip() != '' else 'pending')
    )

no_url_mask = df['pdf_url'].isna() | (df['pdf_url'].astype(str).str.strip().isin(['-', '', 'nan']))
df.loc[no_url_mask, 'dl_status'] = 'no_url'

# ── Build task list (everything not yet done) ─────────────────────────────────
pending_df = df[~df['dl_status'].isin(['done', 'no_url'])]
print(f'Pending : {len(pending_df):,}  |  '
      f'Done : {(df["dl_status"]=="done").sum():,}  |  '
      f'No URL : {no_url_mask.sum():,}  |  '
      f'Total : {len(df):,}')

if len(pending_df) == 0:
    print('🎉 All done! Nothing left to download.')
else:
    tasks = []
    for idx, row in pending_df.iterrows():
        url      = str(row['pdf_url']).strip()
        sym      = safe(row.get('symbol', 'UNKNOWN'))
        seq      = safe(str(row.get('seq_id', '')).split('.')[0])
        url_base = os.path.splitext(os.path.basename(url))[0][:50]
        base_name = f'{sym}_{seq}_{url_base}'
        dest_dir  = os.path.join(OUT_DIR, sym)  # one sub-folder per symbol
        tasks.append((idx, url, dest_dir, base_name))

    ok = skip = fail = counter = 0
    _stop = threading.Event()
    _lock = threading.Lock()

    # Collect failed URLs for separate CSV
    _failed_rows = []

    def _autosave():
        while not _stop.wait(TIMER_SAVE):
            with _lock:
                try: df.to_csv(MANIFEST_CSV, index=False, quoting=1)
                except: pass
    threading.Thread(target=_autosave, daemon=True, name='autosave').start()

    try:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            futs = {
                pool.submit(download_row, u, d, b): (i, u)
                for i, u, d, b in tasks
            }
            pbar = tqdm(total=len(futs), desc='NSE Dividend Docs', unit='file')
            for fut in as_completed(futs):
                i, u = futs[fut]
                try:    res = fut.result()
                except: res = 'thread_crash'

                if res in ('ok', 'skip'):
                    df.at[i, 'dl_status'] = 'done'
                    if res == 'ok':   ok   += 1
                    else:             skip += 1
                else:
                    fail += 1
                    df.at[i, 'dl_status'] = res  # stores http_404 / fail / bad_content
                    _failed_rows.append({'index': i, 'url': u, 'status': res,
                                         'timestamp': datetime.now().isoformat()})
                    if fail <= 10:
                        print(f'\n  [DEBUG] Failed: {u[:70]}... Reason: {res}')

                counter += 1

                # Save manifest periodically
                if counter % SAVE_EVERY == 0:
                    with _lock:
                        df.to_csv(MANIFEST_CSV, index=False, quoting=1)

                # ADDITIONAL: Re-warm session every REWARM_EVERY downloads
                if counter % REWARM_EVERY == 0:
                    print(f'\n[{datetime.now():%H:%M:%S}] Re-warming session after {counter} files...')
                    warm_up_session()

                # ADDITIONAL: Print timestamp every 50 files
                if counter % 50 == 0:
                    print(f'  [{datetime.now():%H:%M:%S}] Progress: {counter}/{len(futs)} | ok={ok} skip={skip} fail={fail}')

                # BUG 4 FIX: Pause & re-warm if >50% bad_content
                if counter % 10 == 0 and _should_pause_and_rewarm():
                    print(f'\n⚠️  [{datetime.now():%H:%M:%S}] >50% bad_content detected — pausing 60s and re-warming...')
                    time.sleep(60)
                    warm_up_session()
                    with _recent_lock:
                        _recent_results.clear()

                pbar.set_postfix(ok=ok, skip=skip, fail=fail, refresh=False)
                pbar.update(1)
            pbar.close()

    except KeyboardInterrupt:
        print('\n⚠️  Interrupted — saving manifest before exit...')
    finally:
        _stop.set()
        with _lock:
            df.to_csv(MANIFEST_CSV, index=False, quoting=1)
        print(f'💾 Manifest saved → {MANIFEST_CSV}')

        # ADDITIONAL: Save failed URLs to separate CSV
        if _failed_rows:
            try:
                pd.DataFrame(_failed_rows).to_csv(FAILED_CSV, index=False)
                print(f'📋 Failed URLs saved → {FAILED_CSV}')
            except:
                pass

    total    = len(df)
    done_n   = (df['dl_status'] == 'done').sum()
    no_url_n = (df['dl_status'] == 'no_url').sum()
    pending_n = total - done_n - no_url_n

    sep = '=' * 50
    print(f'\n{sep}')
    print(f'  SUMMARY')
    print(f'  Total rows     : {total:,}')
    print(f'  Done (total)   : {done_n:,}')
    print(f'    └ New OK     : {ok:,}')
    print(f'    └ Skipped    : {skip:,}')
    print(f'  Failed         : {fail:,}')
    print(f'  No URL         : {no_url_n:,}')
    print(f'  Still pending  : {pending_n:,}')
    print(f'{sep}')
    if fail > 0:
        print(f'\n💡 Tip: Re-run Cell 4 to retry the {fail:,} failed files.')
        failed_codes = df[~df['dl_status'].isin(['done','no_url','pending'])]['dl_status'].value_counts()
        print('  Failure breakdown:')
        for code, cnt in failed_codes.items(): print(f'    {code}: {cnt:,}')

[05:01:40] Warming up NSE session...
  www.nseindia.com → 403
  ⚠️  www.nseindia.com failed (403 Client Error: Forbidden for url: https://www.nseindia.com/) — triggering fallback...
  nsearchives fallback → 200
  ✅ Session warmed via nsearchives fallback
Pending : 134  |  Done : 7,051  |  No URL : 6,264  |  Total : 13,449


NSE Dividend Docs:   0%|          | 0/134 [00:00<?, ?file/s]

NSE Dividend Docs:  26%|██▌       | 35/134 [04:33<12:53,  7.82s/file, fail=17, ok=0, skip=18]



  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/BM_Outcome_VOFL_06012015092... Reason: http_404
  [05:02:28] Progress: 50/134 | ok=24 skip=25 fail=1

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/SSWL_divi_23102015120034.zi... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/BM_Outcome_15-Mar-2016_1503... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/Announcement10032016_100320... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/BM_intimation_29022016_2902... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/Escorts_FYresults_31032016_... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/ASHOY_25052016141728.zip... Reason: unzip_fail

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/Bajajauto_25052016134154.zi... Reason: http_404

  [DEBUG] Failed: https://nsearchives.nseindia.com/corporate/TECHM_